Descrierea de pe site:

The Monty Hall Problem is a very famous problem in Probability Theory. The question goes like:

Suppose you’re on a game show, and you’re given the choice of three doors: Behind one door is a car; behind the others, goats. You pick a door, say No. 1, and the host, who knows what’s behind the doors, opens another door, say No. 3, which has a goat. He then says to you, “Do you want to pick door No. 2?” Is it to your advantage to switch your choice?

By intution it seems that there shouldn’t be any benefit of switching the door. But using Bayes’ Theorem we can show that by switching the door the contestant has more chances of winning.

Paradoxul constă din faptul că deși ar părea că nu contează dacă rămânem pe aceeași ușă sau schimbăm, este de două ori mai probabil să nimerim premiul dacă schimbăm ușa.

Intuitiv, acest fenomen poate fi explicat de faptul că inițial avem o șansă de $\frac 13$ ca să alegem ușa corectă, iar acest lucru nu este influențat de alegerea ulterioară a host-ului.

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

# Defining the network structure
model = DiscreteBayesianNetwork([("C", "H"), ("P", "H")])

# Defining the CPDs:
cpd_c = TabularCPD("C", 3, [[0.33], [0.33], [0.33]]) # probabilitatea ca concurentul sa aleaga fiecare usa
cpd_p = TabularCPD("P", 3, [[0.33], [0.33], [0.33]]) # probabilitatea ca premiul sa fie in spatele fiecarei usi
cpd_h = TabularCPD(
    "H",
    3,
    [
        [0, 0, 0, 0, 0.5, 1, 0, 1, 0.5], # P(H=0 | C, P), matricea distributiei comune C,P este comprimata intr-o lista
        [0.5, 0, 1, 0, 0, 0, 1, 0, 0.5], # P(H=1 | C, P), matricea distributiei comune C,P este comprimata intr-o lista
        [0.5, 1, 0, 1, 0.5, 0, 0, 0, 0], # P(H=2 | C, P), matricea distributiei comune C,P este comprimata intr-o lista
    ],
    evidence=["C", "P"], # variabilele aleatoare de care depinde H
    evidence_card=[3, 3], # numarul de outcome-uri pentru C, respectiv P
)

# Associating the CPDs with the network structure.
model.add_cpds(cpd_c, cpd_p, cpd_h)

# Some other methods
model.get_cpds()

# check_model check for the model structure and the associated CPD and returns True if everything is correct otherwise throws an exception
model.check_model()

# Infering the posterior probability
from pgmpy.inference import VariableElimination

infer = VariableElimination(model)
posterior_p = infer.query(["P"], evidence={"C": 0, "H": 2}) # posterior_p = P(P | C, H)
print(posterior_p)